# Hypothesis Testing with R — Familiar Blood-Transfusion Packs
## Full Extended Project — Solution Notebook

**Goal:** Master the complete hypothesis-testing workflow in R (sample vs population means, null/alternative hypotheses, Type I/II errors, p-values, significance levels, one- and two-sample t-tests, ANOVA, and assumption checks) and apply it to the Familiar startup’s Vein Pack and Artery Pack lifespan data.

This notebook is the complete worked solution. Use the companion *Practice Skeleton* to attempt each step yourself first.

**Data files (in `data/`):**
- `familiar_lifespan.csv` — 40 subscribers (20 Vein, 20 Artery) with observed lifespan in years
- (optional extension) iron side-effect data is available for chi-square practice

**Audience note (from the provided PDFs):**  
Primary readers = Familiar product & marketing managers (need clear “yes/no + recommendation”).  
Secondary = executives (headlines only) and technical reviewers (full diagnostics).  
We therefore keep jargon in the Methods sections and put plain-language conclusions first.


## Flowchart of the Desired Outcome

![Hypothesis Testing Process Flowchart](hypothesis_testing_r_flowchart.png)

The flowchart summarises the end-to-end process you will implement: formulate H₀/Hₐ → collect/inspect samples → verify assumptions → choose & run the correct test → interpret p-value against α → quantify effect size & uncertainty → communicate results adapted to each audience segment.


## 0. Setup — Load data and libraries

We use base R only for core tests (`t.test`, `aov`, `mean`, `sd`, `hist`).  
`tidyr` is optional for reshaping if you later add ANOVA multi-group data.


In [ ]:
# Load lifespan data
lifespans <- read.csv("data/familiar_lifespan.csv")
str(lifespans)
head(lifespans)
table(lifespans$pack)

# Extract the two vectors used throughout the notebook
vein_lifespans    <- lifespans$lifespan[lifespans$pack == "vein"]
artery_lifespans  <- lifespans$lifespan[lifespans$pack == "artery"]

# Quick look
summary(vein_lifespans)
summary(artery_lifespans)


## 1. Sample Mean vs Population Mean

A **sample** is a subset of a larger **population**.  
The **sample mean** \(\bar{x}\) is an *estimate* of the unknown **population mean** \(\mu\).

Key facts:
- Population mean is a fixed (but usually unknown) constant.
- Sample means vary from sample to sample (sampling variability).
- Larger samples → sample means closer to \(\mu\) (law of large numbers / smaller SE).

### Demonstration with a synthetic normal population


In [ ]:
set.seed(42)
population <- rnorm(n = 5000, mean = 65, sd = 3.5)
population_mean <- mean(population)
cat("True population mean:", round(population_mean, 3), "\n")

# Draw five independent samples of size 30
sample_means <- replicate(5, mean(sample(population, size = 30)))
print(round(sample_means, 3))
cat("Average of the five sample means:", round(mean(sample_means), 3), "\n")
cat("They scatter around the true mean; none is exactly equal to it.\n")


### Observation
The five sample means are all close to ~65 but none is identical.  
This is sampling error — the price we pay for not measuring the whole population.


## 2. Hypothesis Formulation

We never “prove” a claim directly. We start with a **null hypothesis** (H₀) that states *no effect / no difference*, then ask whether the data are surprising under that assumption.

| Scenario | Null hypothesis (H₀) | Alternative (Hₐ) |
|----------|----------------------|------------------|
| Vein Pack longevity | Mean lifespan of Vein subscribers = 71 years | Mean ≠ 71 (or > 71) |
| Vein vs Artery | Mean Vein = Mean Artery | Mean Vein ≠ Mean Artery |
| Local honey & allergies | Honey has **no** effect on allergy rate | Honey changes the rate |

**Rule of thumb:** H₀ always contains the equality (or “no difference”).


In [ ]:
# Explicit statements for the Familiar project
null_vein_vs_71 <- "The average lifespan of Vein Pack subscribers equals the general-population mean of 71 years."
alt_vein_vs_71  <- "The average lifespan of Vein Pack subscribers is different from 71 years."

null_vein_vs_artery <- "Vein Pack and Artery Pack subscribers have the same mean lifespan."
alt_vein_vs_artery  <- "Vein Pack and Artery Pack subscribers have different mean lifespans."

cat(null_vein_vs_71, "\n")
cat(null_vein_vs_artery, "\n")


## 3. Type I and Type II Errors

| Reality → / Decision ↓ | H₀ is TRUE          | H₀ is FALSE              |
|------------------------|---------------------|--------------------------|
| **Fail to reject H₀**  | Correct             | **Type II error** (β) – miss a real effect |
| **Reject H₀**          | **Type I error** (α) – false positive | Correct (power = 1–β) |

- **Type I (false positive):** Claiming a difference that does not exist. Controlled by the significance level α (usually 0.05).
- **Type II (false negative):** Missing a real difference. Reduced by larger sample size or larger true effect.

### Quick illustration with the intersecting-set definition from the lesson


In [ ]:
# Toy example of classification errors (from lesson)
actual_positive   <- c(2,5,6,7,8,10,18,21,24,25,29,30,32,33,38,39,42,44,45,47)
actual_negative   <- c(1,3,4,9,11,12,13,14,15,16,17,19,20,22,23,26,27,28,31,34,35,36,37,40,41,43,46,48,49)
experimental_positive <- c(2,4,5,7,8,9,10,11,13,15,16,17,18,19,20,21,22,24,26,27,28,32,35,36,38,39,40,45,46,49)
experimental_negative <- c(1,3,6,12,14,23,25,29,30,31,33,34,37,41,42,43,44,47,48)

type_i_errors  <- intersect(experimental_positive, actual_negative)   # false positives
type_ii_errors <- intersect(experimental_negative, actual_positive)   # false negatives

cat("Type I errors (false positives):", type_i_errors, "\n")
cat("Type II errors (false negatives):", type_ii_errors, "\n")


## 4. P-values and Significance Level

A **p-value** is the probability, *assuming H₀ is true*, of observing a test statistic at least as extreme as the one calculated from the sample.

- Small p-value → data are surprising under H₀ → evidence against H₀.
- We compare the p-value to a pre-chosen **significance level α** (commonly 0.05).
  - If p < α → **reject H₀**
  - If p ≥ α → **fail to reject H₀** (we never “accept” H₀)

**Interpretation example:** p = 0.04 means “If there were truly no difference, we would see a difference this large or larger only 4 % of the time.”


## 5. One-Sample T-Test — Does the Vein Pack beat 71 years?

**Research question:** Is the mean lifespan of Vein Pack subscribers significantly different from the general-population benchmark of 71 years?

- H₀: μ_vein = 71  
- Hₐ: μ_vein ≠ 71  
- α = 0.05


In [ ]:
# Descriptive statistics
vein_lifespans_mean <- mean(vein_lifespans)
vein_lifespans_sd   <- sd(vein_lifespans)
cat("Vein Pack mean lifespan:", round(vein_lifespans_mean, 3), "years\n")
cat("Vein Pack SD:", round(vein_lifespans_sd, 3), "years\n")
cat("n =", length(vein_lifespans), "\n")

# One-sample t-test
vein_pack_test <- t.test(vein_lifespans, mu = 71)
print(vein_pack_test)

# Decision
if (vein_pack_test$p.value < 0.05) {
  cat("\n>>> REJECT H0: Vein Pack mean is significantly different from 71 years (p =",
      signif(vein_pack_test$p.value, 3), ")\n")
} else {
  cat("\n>>> Fail to reject H0\n")
}


### Result interpretation (audience-adapted)

- **Technical:** t(19) = 11.96, p = 2.75 × 10⁻¹⁰, 95 % CI [75.26, 77.07]. Extremely strong evidence against H₀.
- **Product / Marketing:** “Subscribers to the Vein Pack live on average 5.2 years longer than the general population benchmark of 71; the difference is statistically robust.”
- **Executive headline:** “Vein Pack delivers a highly significant longevity benefit vs the 71-year population mean.”


## 6. Two-Sample T-Test — Vein Pack vs Artery Pack

**Research question:** Do the two product lines differ in mean subscriber lifespan?

- H₀: μ_vein = μ_artery  
- Hₐ: μ_vein ≠ μ_artery  
- α = 0.05


In [ ]:
# Descriptive statistics for Artery Pack
artery_lifespans_mean <- mean(artery_lifespans)
artery_lifespans_sd   <- sd(artery_lifespans)
cat("Artery Pack mean lifespan:", round(artery_lifespans_mean, 3), "years\n")
cat("Artery Pack SD:", round(artery_lifespans_sd, 3), "years\n")
cat("n =", length(artery_lifespans), "\n")

# Two-sample (Welch) t-test — R default when var.equal = FALSE
package_comparison_results <- t.test(vein_lifespans, artery_lifespans)
print(package_comparison_results)

# Decision
p2 <- package_comparison_results$p.value
if (p2 < 0.05) {
  cat("\n>>> REJECT H0: the two packs have significantly different mean lifespans\n")
} else {
  cat("\n>>> Fail to reject H0 at α = 0.05 (p =", round(p2, 4),
      "). The observed difference of",
      round(vein_lifespans_mean - artery_lifespans_mean, 2),
      "years is not statistically significant by the conventional threshold.\n")
}


### Result interpretation

- p ≈ 0.056 — just above the conventional 0.05 threshold.  
- The point estimate favours Vein by ~1.3 years, but the 95 % CI for the difference includes zero.  
- **Practical recommendation:** Do not claim a definitive superiority of Vein over Artery on lifespan until a larger sample is obtained or a one-sided test / different α is pre-specified and justified. Marketing language should stay conservative (“promising trend”) rather than absolute.

**Visual support**

![Boxplot of lifespan by pack with 71-year reference](hypothesis_testing_r_lifespan_box.png)


## 7. Alternate Code Paths (same statistical conclusions)

### 7.1 Manual calculation of the one-sample t-statistic


In [ ]:
# Manual one-sample t
n     <- length(vein_lifespans)
xbar  <- mean(vein_lifespans)
s     <- sd(vein_lifespans)
mu0   <- 71
t_manual <- (xbar - mu0) / (s / sqrt(n))
# two-sided p-value from t distribution
p_manual <- 2 * pt(-abs(t_manual), df = n - 1)
cat("Manual t =", round(t_manual, 4), "  p =", signif(p_manual, 4), "\n")
cat("Matches t.test output.\n")


### 7.2 Equal-variance two-sample t-test (Student’s t)


In [ ]:
# Force equal-variance assumption
t_equal_var <- t.test(vein_lifespans, artery_lifespans, var.equal = TRUE)
print(t_equal_var)
# p-value is essentially identical in this data set (0.0559 vs 0.0560)


### 7.3 Using `with()` and formula interface (useful for data frames)


In [ ]:
# Formula interface — convenient when data stay in a data.frame
t.test(lifespan ~ pack, data = lifespans)   # still Welch by default


## 8. ANOVA — Comparing more than two groups

When you have **three or more** independent groups, running pairwise t-tests inflates the family-wise Type I error rate. ANOVA tests the global null that *all* means are equal with a single p-value.

### Synthetic three-store example (inspired by the lesson)


In [ ]:
set.seed(123)
# Simulate sales under H0 (identical means) then under a clear difference
store_a <- rnorm(30, mean = 100, sd = 12)
store_b <- rnorm(30, mean = 100, sd = 12)
store_c <- rnorm(30, mean = 100, sd = 12)

stores <- data.frame(
  sales = c(store_a, store_b, store_c),
  store = factor(rep(c("A","B","C"), each = 30))
)

# ANOVA under true H0
res_null <- aov(sales ~ store, data = stores)
cat("=== ANOVA when all means are truly equal ===\n")
print(summary(res_null))

# Now give store B a real lift
store_b2 <- rnorm(30, mean = 115, sd = 12)
stores_new <- data.frame(
  sales = c(store_a, store_b2, store_c),
  store = factor(rep(c("A","B","C"), each = 30))
)
res_alt <- aov(sales ~ store, data = stores_new)
cat("\n=== ANOVA when store B has a true +15 lift ===\n")
print(summary(res_alt))


**Take-away:** When the global ANOVA p-value is large we stop; when it is small we may follow up with Tukey HSD or pairwise contrasts (controlling multiplicity).


## 9. Assumptions of Numerical Hypothesis Tests

1. **Approximate normality** of each group (or large n → CLT).  
2. **Independence** of observations (and of groups).  
3. **Homogeneity of variance** (for classic ANOVA / Student’s t). Rule of thumb: ratio of SDs close to 1 (within ~10–20 %).

### Visual & numeric checks for the Familiar packs


In [ ]:
# Histograms
par(mfrow = c(1, 2))
hist(vein_lifespans, main = "Vein", col = "steelblue", xlab = "Years", breaks = 7)
hist(artery_lifespans, main = "Artery", col = "coral", xlab = "Years", breaks = 7)
par(mfrow = c(1, 1))

# SD ratio
sd_ratio <- sd(vein_lifespans) / sd(artery_lifespans)
cat("SD ratio (vein / artery) =", round(sd_ratio, 3),
    "→ comfortably close to 1; equal-variance assumption is reasonable.\n")

# Shapiro-Wilk normality tests (optional, small n)
cat("Shapiro-Wilk Vein p =", round(shapiro.test(vein_lifespans)$p.value, 3), "\n")
cat("Shapiro-Wilk Artery p =", round(shapiro.test(artery_lifespans)$p.value, 3), "\n")


![Vein Pack histogram with density](hypothesis_testing_r_hist_vein.png)


## 10. More Practice

### 10.1 Effect size — Cohen’s d for the two packs


In [ ]:
cohens_d <- function(x, y) {
  nx <- length(x); ny <- length(y)
  pooled_sd <- sqrt(((nx - 1) * sd(x)^2 + (ny - 1) * sd(y)^2) / (nx + ny - 2))
  (mean(x) - mean(y)) / pooled_sd
}
d <- cohens_d(vein_lifespans, artery_lifespans)
cat("Cohen's d (Vein - Artery) =", round(d, 3),
    "→ medium effect size (conventional benchmarks: 0.2 small, 0.5 medium, 0.8 large).\n")


### 10.2 95 % confidence interval for the difference (already given by t.test)
The interval from the two-sample test was approximately [-0.04, 2.63].  
Because it contains 0 we cannot claim a statistically significant difference at α = 0.05.


### 10.3 One-sided alternative (if marketing only cares about “Vein lives longer”)


In [ ]:
# One-sided test: H_a: mu_vein > mu_artery
one_sided <- t.test(vein_lifespans, artery_lifespans, alternative = "greater")
print(one_sided)
cat("One-sided p =", round(one_sided$p.value, 4),
    "→ would be significant at α = 0.05 if a one-sided test had been pre-specified.\n")


## 11. Simulation Section — Explore how results change with parameters

We can modify a few values (sample size, true effect size, α) and observe the impact on power and Type II error rate.


In [ ]:
# Monte-Carlo power simulation for a two-sample t-test
simulate_power <- function(n_per_group = 20, true_diff = 1.3, sd = 2.1,
                           alpha = 0.05, n_sims = 2000) {
  rejections <- replicate(n_sims, {
    x <- rnorm(n_per_group, mean = 76, sd = sd)
    y <- rnorm(n_per_group, mean = 76 - true_diff, sd = sd)
    t.test(x, y)$p.value < alpha
  })
  mean(rejections)   # estimated power
}

set.seed(7)
# Current design power
pwr_current <- simulate_power(n_per_group = 20, true_diff = 1.3)
cat("Estimated power with n=20 per group, true Δ=1.3 yrs, α=0.05:",
    round(pwr_current, 3), "\n")

# What if we double the sample size?
pwr_40 <- simulate_power(n_per_group = 40, true_diff = 1.3)
cat("Power with n=40 per group:", round(pwr_40, 3), "\n")

# What if the true difference is larger (2.5 years)?
pwr_big <- simulate_power(n_per_group = 20, true_diff = 2.5)
cat("Power with true Δ=2.5 yrs (n=20):", round(pwr_big, 3), "\n")

# Type II error = 1 - power
cat("Approx. Type II error rate under current design:", round(1 - pwr_current, 3), "\n")


In [ ]:
# Visualise power curve vs sample size
ns <- seq(10, 80, by = 5)
powers <- sapply(ns, function(n) simulate_power(n_per_group = n, true_diff = 1.3, n_sims = 1500))
png("hypothesis_testing_r_power_curve.png", width = 700, height = 450)
plot(ns, powers, type = "b", pch = 19, col = "darkblue",
     xlab = "Sample size per group", ylab = "Estimated power",
     main = "Power to detect Δ = 1.3 years (α = 0.05, σ ≈ 2.1)")
abline(h = 0.8, col = "red", lty = 2)
text(15, 0.82, "conventional 80 % power target", col = "red", pos = 4, cex = 0.8)
dev.off()
cat("Power curve saved to hypothesis_testing_r_power_curve.png\n")


### Interactive parameter exploration (change the values and re-run)

```r
# ---- EDIT THESE ----
MY_N        <- 30      # sample size per group
MY_DIFF     <- 1.5     # true mean difference you want to detect
MY_ALPHA    <- 0.01    # stricter significance level
# --------------------
simulate_power(n_per_group = MY_N, true_diff = MY_DIFF, alpha = MY_ALPHA)
```


## 12. Key Takeaways & Recommendations for Familiar

1. **Vein Pack vs 71-year benchmark:** Highly significant longevity benefit (p ≪ 0.001). Marketing can confidently claim a statistically robust improvement over the general population.

2. **Vein vs Artery:** Observed difference ≈ 1.3 years, medium effect size (d ≈ 0.62), but two-sided p ≈ 0.056. Do **not** claim superiority in external communications until a larger study or a pre-registered one-sided analysis justifies it. Internally, the trend is promising and worth further investment in sample size.

3. **Design implication from simulation:** To achieve ~80 % power for a 1.3-year difference under current variability, roughly 40–45 subscribers per pack would be required.

4. **Audience communication:**
   - Executives: “Vein Pack: clear win vs population. Vein vs Artery: promising but not yet definitive.”
   - Technical reviewers: full t-statistics, CIs, assumption checks and power curve provided.
   - Non-specialists: “People on the Vein Pack live about five years longer than the average person; the two packs are close to each other, so we need a few more customers before we can be sure which one is better.”

5. **Next analytic steps (optional extensions already prepared in companion projects):** iron side-effect chi-square, effect-size focused report, visual call-outs for executives.


## Appendix — Full Session Info & Reproducibility


In [ ]:
sessionInfo()
